In [1]:
import pandas as pd
import numpy as np

In [2]:
from bracket_functions import create_initial_bracket, get_teams
from bracket_predictions import sequential_predictions, parallel_predictions
import bracket_predictions
import json
from scipy.stats import norm

In [3]:
YEAR = 2022
DATA_DIRECTORY = f"data/{YEAR}"
DATA_FILE_NAME = f'{DATA_DIRECTORY}/simple_{YEAR}.csv'
ESPN_FILE = (
    'https://projects.fivethirtyeight.com/'
    f'march-madness-api/{YEAR}/fivethirtyeight_ncaa_forecasts.csv'
)

In [4]:
ORDER_OF_REGIONS = ['West', 'East', 'South', 'Midwest']  # This changes every year!

In [5]:
initial_bracket, data = create_initial_bracket(DATA_FILE_NAME, ORDER_OF_REGIONS)

In [6]:
# Import 538's info
espn = pd.read_csv(ESPN_FILE)
espn_slice = (espn['gender'] == 'mens') & (espn['forecast_date'] == max(espn['forecast_date']))
team_info = espn[espn_slice][['team_id', 'team_name', 'team_region', 'team_seed']]

In [7]:
espn_data = espn[espn_slice]
espn_data.head()

,gender,forecast_date,playin_flag,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,results_to,team_alive,team_id,team_name,team_rating,team_region,team_seed,team_slot
0,mens,2022-04-02,0,1.0,1.0,1.0,1.0,1.0,1.0,0.677544,6,1,2305,Kansas,95.34,Midwest,1,64
1,mens,2022-04-02,0,1.0,1.0,1.0,1.0,1.0,1.0,0.322456,6,1,153,North Carolina,91.24,East,8,36
2,mens,2022-04-02,0,1.0,1.0,1.0,1.0,1.0,0.0,0.000000,7,0,150,Duke,92.67,West,2,28
3,mens,2022-04-02,0,1.0,1.0,1.0,1.0,1.0,0.0,0.000000,7,0,222,Villanova,89.94,South,2,124
4,mens,2022-04-02,0,1.0,1.0,1.0,1.0,0.0,0.0,0.000000,7,0,8,Arkansas,89.96,West,4,12


In [8]:
# Set championship result
espn_data.loc[0, "rd7_win"] = 1
espn_data.loc[1, "rd7_win"] = 0

/Users/Alex/opt/anaconda3/envs/py3/lib/python3.8/site-packages/pandas/core/indexing.py:966: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[item] = s


In [9]:
data.head()

,64,team_region,Seed,team_id,AdjustEM,AdjustT
0,Gonzaga,West,1,2250.0,32.95,72.5
1,Georgia State,West,16,2247.0,1.66,67.1
2,Boise State,West,8,68.0,16.13,64.6
3,Memphis,West,9,235.0,16.18,70.3
4,Connecticut,West,5,41.0,19.27,64.9


In [10]:
def create_specific_bracket(initial_bracket, data, espn, bracket_type_func):
    bracket = initial_bracket.copy()
    rounds = ['64', '32', '16', '8', '4', '2', '1']
    for round_ in range(6):
        gap = 2**(round_ + 1)  # how far apart two teams could be for a given round
        for top_team in range(0, len(bracket), gap):
            pyths = get_teams(top_team, gap, bracket, rounds, round_, data)
            winner_num, winner_name = bracket_type_func(pyths, data, espn, round_, bracket)
            bracket.loc[winner_num, rounds[round_ + 1]] = winner_name  # write in the winner
    return bracket

In [11]:
def get_team_ids(pyths, data):
    return data.loc[data["64"].isin([i[1] for i in pyths])]["team_id"].tolist()


def get_winner(team_ids, data, round_):
    df = data.loc[lambda x: x["team_id"].isin(team_ids)]
    return df.loc[df[f"rd{round_ + 2}_win"].idxmax()]["team_name"]


def perfect_bracket(pyths, data, espn, round_, *args):
    team_ids = get_team_ids(pyths, data)
    winner_name = get_winner(team_ids, espn, round_)
    winner_num = [i[0] for i in pyths if i[1] == winner_name][0]
    return winner_num, winner_name

In [12]:
def compute_prob(pyths):
    A_team_id = pyths[0][0]
    A_team_name = pyths[0][1]
    A_em = pyths[0][2]
    A_t = pyths[0][3]
    B_team_id = pyths[1][0]
    B_team_name = pyths[1][1]
    B_em = pyths[1][2]
    B_t = pyths[1][3]

    point_diff = (A_em - B_em) * (A_t + B_t) / 200

    d = norm(0, 11)
    prob = d.cdf(point_diff)
    return prob, A_team_id, A_team_name, B_team_id, B_team_name

In [13]:
def higher_prob(pyths, *args):
    prob, A_team_id, A_team_name, B_team_id, B_team_name = compute_prob(pyths)
    if prob >= 0.5:
        return A_team_id, A_team_name
    else:
        return B_team_id, B_team_name

In [14]:
def lower_prob(pyths, *args):
    prob, A_team_id, A_team_name, B_team_id, B_team_name = compute_prob(pyths)
    if prob >= 0.5:
        return B_team_id, B_team_name
    else:
        return A_team_id, A_team_name

In [15]:
def get_seeds(pyths, bracket):
    seed_A = bracket.loc[bracket["64"] == pyths[0][1], "Seed"].values[0]
    seed_B = bracket.loc[bracket["64"] == pyths[1][1], "Seed"].values[0]
    return seed_A, seed_B


def chalk(pyths, bracket, *args):
    seed_A, seed_B = get_seeds(pyths, bracket)
    if seed_A <= seed_B:
        return pyths[0][0], pyths[0][1]
    else:
        return pyths[1][0], pyths[1][1]

In [16]:
def compute_winner_prob(pyths, winner_id):
    prob, A_team_id, A_team_name, B_team_id, B_team_name = compute_prob(pyths)
    
    if A_team_id == winner_id:
        return prob
    elif B_team_id == winner_id:
        return 1 - prob
    else:
        raise Exception("Hmm")

In [17]:
def compute_bracket_probability(bracket):
    rounds = ['64', '32', '16', '8', '4', '2', '1']
    probs = []
    for round_ in range(6):
        gap = 2**(round_ + 1)  # how far apart two teams could be for a given round
        for top_team in range(0, len(bracket), gap):
            pyths = get_teams(top_team, gap, bracket, rounds, round_, data)
            winner_id = get_teams(top_team, gap, bracket, rounds, round_ + 1, data)[0][0]
            prob = compute_winner_prob(pyths, winner_id)
            probs.append(prob)

    full_prob = np.prod(probs)  # compute the likelihood of the entire bracket
    return full_prob

In [35]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, higher_prob)
prob = compute_bracket_probability(bracket)
prob

7.464989580978435e-12

In [36]:
1/prob

133958659841.68329

In [33]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, perfect_bracket)
prob = compute_bracket_probability(bracket)
prob

5.741301662958249e-18

In [34]:
1/prob

1.7417652976707418e+17

In [31]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, lower_prob)
prob = compute_bracket_probability(bracket)
prob

2.2088276171324476e-38

In [32]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, chalk)
prob = compute_bracket_probability(bracket)
prob

1.3728409519649738e-12

In [23]:
display(bracket.loc[bracket['8'] != ''][["Region", "Seed", "8", "4", "2", "1"]])

,Region,Seed,8,4,2,1
0,West,1,Gonzaga,Gonzaga,Gonzaga,Gonzaga
14,West,2,Duke,,,
16,East,1,Baylor,Baylor,,
30,East,2,Kentucky,,,
32,South,1,Arizona,Arizona,Arizona,
46,South,2,Villanova,,,
48,Midwest,1,Kansas,Kansas,,
62,Midwest,2,Auburn,,,
